# ECG Heartbeat Classification Project
## Phase 5 Explainable AI (SHAP and LIME)

This notebook applies Explainable AI techniques to the trained
Random Forest model to understand and interpret its predictions.

SHAP (Shapley Additive explanations) explains which features
contributed most to predictions both globally across all beats
and locally for individual predictions.

LIME (Local Interpretable Model-agnostic Explanations) provides
an alternative local explanation for individual predictions by
approximating the model behaviour around a specific beat.

Together SHAP and LIME make the model transparent and trustworthy
for clinical use, addressing the black box problem in medical AI.

### Import libraries and load saved model
Loading the trained Random Forest model saved in Phase 4
along with the feature dataset for explanation analysis.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')

# Load the saved Random Forest model from Phase 4
rf_model = joblib.load('../models/random_forest_model.pkl')
print("Random Forest model loaded")

# Load feature names
with open('../models/feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f.readlines()]
print(f"Feature names loaded: {feature_names}")

# Load the feature dataset
df = pd.read_csv('../data/ecg_features.csv')

# Separate features and labels
X = df.drop('label', axis=1)
y = df['label']

# Use the same split as Phase 4 for consistency
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nDataset loaded successfully")
print(f"Training set: {X_train.shape[0]:,} beats")
print(f"Testing set:  {X_test.shape[0]:,} beats")
print(f"Feature count: {X_test.shape[1]}")

Random Forest model loaded
Feature names loaded: ['mean', 'std', 'max', 'min', 'range', 'r_peak_amplitude', 'skewness', 'kurtosis', 'qrs_width', 'rr_interval', 'mean_first_half', 'mean_second_half', 'half_difference']

Dataset loaded successfully
Training set: 81,905 beats
Testing set:  20,477 beats
Feature count: 13


### SHAP Global Explanation
SHAP (Shapley Additive explanations) calculates how much each
feature contributed to the model predictions across all test
beats. The global explanation shows which features the model
relied on most overall — providing a bird's eye view of the
model's decision making process.

In [7]:
print("Calculating SHAP values:")

# Create SHAP explainer for Random Forest
# TreeExplainer is specifically designed for tree based models
explainer = shap.TreeExplainer(rf_model)

# Calculate SHAP values for a sample of 1000 beats
# 1000 beats is representative and much faster than all 20,477
sample_size = 1000
X_test_sample = X_test.iloc[:sample_size]
y_test_sample = y_test.iloc[:sample_size]

shap_values = explainer.shap_values(X_test_sample)

print(f"SHAP values calculated for {sample_size} test beats")
print(f"SHAP values shape: {np.array(shap_values).shape}")

# Handle the shape correctly
# shap_values shape is (1000, 13, 2) for binary classification
# We need values for class 1 (Abnormal)
shap_array = np.array(shap_values)

if len(shap_array.shape) == 3:
    # Shape is (n_samples, n_features, n_classes)
    shap_vals_abnormal = shap_array[:, :, 1]
else:
    # Shape is (n_classes, n_samples, n_features)
    shap_vals_abnormal = shap_array[1]

print(f"Adjusted SHAP values shape: {shap_vals_abnormal.shape}")

# Calculate mean absolute SHAP value per feature
mean_shap = np.abs(shap_vals_abnormal).mean(axis=0)
shap_ranking = np.argsort(mean_shap)[::-1]

# Print ranking
print(f"\nMean absolute SHAP values per feature:")
print(f"{'Rank':<6} {'Feature':<20} {'Mean |SHAP|':>12}")
print("─" * 42)
for rank, idx in enumerate(shap_ranking, 1):
    print(f"{rank:<6} {feature_names[idx]:<20} {mean_shap[idx]:>12.4f}")

Calculating SHAP values:
SHAP values calculated for 1000 test beats
SHAP values shape: (1000, 13, 2)
Adjusted SHAP values shape: (1000, 13)

Mean absolute SHAP values per feature:
Rank   Feature               Mean |SHAP|
──────────────────────────────────────────
1      kurtosis                   0.0809
2      qrs_width                  0.0772
3      skewness                   0.0759
4      std                        0.0684
5      rr_interval                0.0611
6      range                      0.0350
7      half_difference            0.0324
8      r_peak_amplitude           0.0290
9      min                        0.0256
10     max                        0.0188
11     mean_first_half            0.0136
12     mean_second_half           0.0121
13     mean                       0.0115


### SHAP global explanation observations

Kurtosis ranked first with mean SHAP value of 0.0809,
consistent with both EDA analysis (Phase 3) and Random Forest
feature importance (Phase 4). This is the third independent
method confirming kurtosis as the most influential feature.

The top 5 features by SHAP importance are:
1. kurtosis (0.0809) :beat sharpness
2. qrs_width (0.0772) :width of main spike
3. skewness (0.0759) :beat shape symmetry
4. std (0.0684) :signal variation
5. rr_interval (0.0611) :rhythm timing

Mean based features (mean, mean_first_half, mean_second_half)
consistently rank lowest across all three analysis methods,
confirming they contribute least to classification decisions.

The strong consistency between EDA findings, feature importance
and SHAP rankings demonstrates the model is behaving in a
clinically logical and interpretable manner.